In [ ]:
import CONSTANTS
from functions import *
import warnings
import torch
import torch.nn as nn
import math
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from transformer_classes import *
from tqdm import tqdm
from sklearn.metrics import *

warnings.filterwarnings('ignore')

# Transformer

Objective: predict change in BTC price given features

## Prepare data
Create tupled dataset


In [ ]:
RESPONSE = 'close'

In [ ]:
data = pd.read_csv(f'../{fullDataPath('BTC')}')
daily_data = dataSetup(data, trainingColPath='../training_columns.txt')[-LIMIT:]
daily_data

In [ ]:
daily_data[RESPONSE].plot.line(title=f'{COIN} Price', figsize=(12, 6), ylabel='Price (USD)', xlabel='Date')

In [ ]:
daily_data['gradient'].plot.hist()

In [ ]:
daily_data[RESPONSE].describe()

# Load the Transformer Model

Sequence the data to make it predict the next price

## Preprocess the data


In [ ]:
daily_data = transformerDataSetup(daily_data)
daily_data[['sequence', 'next']]

## Load the Model

### Data Setup

In [2]:
DAYS_TO_PREDICT = TEST_DAYS
X_train, X_test, y_train, y_test, _, _ = transformerXTrainYTrain(daily_data, testSize=len(daily_data)-DAYS_TO_PREDICT)
X_train_norm, X_test_norm, y_train_norm, y_test_norm, sequence_scaler, target_scaler = normalize(X_train, X_test, y_train, y_test)
training_cols = trainingCols()

train_stuff = daily_data.loc[X_train_norm.index, training_cols]
test_stuff = daily_data.loc[X_test_norm.index, training_cols]
X_train_norm = pd.concat([X_train_norm, train_stuff], axis=1)
X_test_norm = pd.concat([X_test_norm, test_stuff], axis=1)

NameError: name 'TEST_DAYS' is not defined

In [1]:
trainingCols()

NameError: name 'trainingCols' is not defined

In [ ]:
# Create model with appropriate settings for Bitcoin prediction
model = BaseTransformer(
    d_model=128,
    num_heads=8,
    num_layers=4,
    output_dim=TEST_DAYS,  # Add this line - this is crucial!
    learning_rate=1e-4,
    batch_size=32,
    dropout=0.1,
    mask_value=FILL
)
model.fit(X_train_norm, y_train_norm, epochs=30, validation_data=(X_test_norm, y_test_norm))
torch.save(model, f'../models/{COIN}_model.pth')

In [ ]:
model = torch.load(f'../models/{COIN}_model.pth')

things = X_train_norm
predictions = model.predict(things)
predictions = target_scaler.inverse_transform(predictions)
predictions_df = pd.DataFrame(predictions, index=things.index, columns=[f'Day {i+1}' for i in range(predictions.shape[1])])

In [57]:
predictions_df['close'] = y_train
predictions_df

,Day 1,Day 2,Day 3,Day 4,Day 5,Day 6,Day 7,close
time,,,,,,,,
2024-06-27,62228.984375,58595.476562,58602.531250,61260.429688,64179.804688,60458.894531,59213.039062,"[61615.39, 60313.35, 60885.67, 62668.26, 62830.13, 62040.22, 60145.01]"
2024-06-28,60056.832031,59699.613281,61302.480469,60483.179688,59876.105469,60018.199219,59914.554688,"[60313.35, 60885.67, 62668.26, 62830.13, 62040.22, 60145.01, 57042.14]"
2024-06-29,60246.992188,59851.156250,61021.808594,60482.933594,60157.878906,60642.023438,60201.433594,"[60885.67, 62668.26, 62830.13, 62040.22, 60145.01, 57042.14, 56639.43]"
2024-06-30,60386.648438,59939.941406,60819.121094,60470.964844,60377.886719,61007.578125,60411.527344,"[62668.26, 62830.13, 62040.22, 60145.01, 57042.14, 56639.43, 58244.75]"
2024-07-01,60496.496094,59989.695312,60668.796875,60463.246094,60553.945312,61260.175781,60608.410156,"[62830.13, 62040.22, 60145.01, 57042.14, 56639.43, 58244.75, 55854.09]"
...,...,...,...,...,...,...,...,...
2025-06-15,107363.335938,107422.750000,107787.085938,108052.828125,107291.257812,107809.929688,107908.343750,"[105599.25, 106853.38, 104590.44, 104915.6, 104671.9, 103317.8, 102160.03]"
2025-06-16,107410.601562,107477.625000,107843.648438,108088.140625,107304.460938,107845.687500,107923.828125,"[106853.38, 104590.44, 104915.6, 104671.9, 103317.8, 102160.03, 100996.87]"
2025-06-17,107453.679688,107528.351562,107896.585938,108115.390625,107314.484375,107874.109375,107933.109375,"[104590.44, 104915.6, 104671.9, 103317.8, 102160.03, 100996.87, 105419.39]"


In [ ]:
diff = predictions - y_test
plt.axhline(0.0, color='r', linestyle='--')
diff.plot.line(title='Difference on Prediction and Actual')

In [ ]:
comparison = pd.concat([y_test, predictions], axis=1)
comparison.columns = ['actual', 'prediction']
comparison.plot.line(title='Predicted vs. Actual Value on Validation Month', xlabel='Date', ylabel='Price')

In [ ]:
new = daily_data[RESPONSE].iloc[-30:]
new, next = sequence(new, len(new)-1)
new = pd.DataFrame({'sequences': [new]})
new = pd.DataFrame({""
        'sequences': normalize_sequences(new.iloc[:, 0], sequence_scaler)
    })
model.predict(new, sequence_scaler)[0, 0]

In [ ]:
price = daily_data[RESPONSE].iloc[-1]
starter = daily_data[RESPONSE].iloc[-30:]
predictions = predict_sequence(model, price, starter, sequence_scaler, sequence_length=7)
new_days = pd.date_range(start=daily_data.index[-1] + pd.Timedelta(days=1), periods=len(predictions), freq='D')
predictions_df = pd.DataFrame(predictions, index=new_days, columns=[RESPONSE])
predictions_df.plot.line()
plt.show()

In [ ]:
predictNextNDaysTransformer(daily_data, total_length=365)